# 03 -- IV/2SLS and LIML: polars_reg vs R

This notebook verifies that `polars_reg.iv2sls()` and `polars_reg.liml()` produce
results equivalent to R's `fixest::feols()` (IV mode) and `AER::ivreg()` (LIML).

**Dataset**: `CigarettesSW` from the R `AER` package (48 US states, 2 years = 96 obs).

**Tests**:
1. 2SLS with iid standard errors
2. 2SLS with robust (HC1) standard errors
3. 2SLS with cluster-robust SEs (clustered by state)
4. LIML estimation
5. First-stage F-statistic
6. Weak instrument diagnostics

In [ ]:
import sys, os
import numpy as np
import polars as pl

# Ensure polars_reg is importable
sys.path.insert(0, os.path.abspath("../.."))
import polars_reg as pr

# Verification helpers
sys.path.insert(0, os.path.abspath("."))
from r_helper import load_r_dataset, run_r_regression, compare, compare_scalar, R_EXTRACT

## Load CigarettesSW and create derived columns

In [ ]:
df = load_r_dataset("CigarettesSW", package="AER")
print(f"Raw shape: {df.shape}")
print(df.head(3))

In [ ]:
# Derived columns
df = df.with_columns([
    (pl.col("price") / pl.col("cpi")).alias("rprice"),
    (pl.col("income") / (pl.col("population") * pl.col("cpi"))).alias("rincome"),
    ((pl.col("taxs") - pl.col("tax")) / pl.col("cpi")).alias("tdiff"),
    (pl.col("tax") / pl.col("cpi")).alias("rtax"),
]).with_columns([
    pl.col("rprice").log().alias("lrprice"),
    pl.col("rincome").log().alias("lrincome"),
    pl.col("packs").log().alias("lpacks"),
])

print(f"Final shape: {df.shape}")
print(df.select(["state", "year", "lpacks", "lrprice", "lrincome", "tdiff", "rtax"]).head(5))

In [ ]:
# Save CSV so R scripts can read the same data
csv_path = os.path.abspath("_cigarettes_iv.csv")
df.write_csv(csv_path)
print(f"Saved to {csv_path}")

In [ ]:
# Common R preamble: load CSV + derive columns (safety: recompute in R too)
R_PREAMBLE = f'''
df <- read.csv("{csv_path}")
df$rprice <- df$price / df$cpi
df$rincome <- df$income / (df$population * df$cpi)
df$tdiff <- (df$taxs - df$tax) / df$cpi
df$rtax <- df$tax / df$cpi
df$lrprice <- log(df$rprice)
df$lrincome <- log(df$rincome)
df$lpacks <- log(df$packs)
'''

---
## Test 1: 2SLS with iid standard errors

In [ ]:
# polars_reg
res_2sls_iid = pr.iv2sls(
    "lpacks ~ lrincome || lrprice ~ tdiff + rtax",
    data=df,
)
res_2sls_iid.summary()

In [ ]:
# R: fixest feols with IV, iid vcov
r_2sls_iid = run_r_regression(f'''
library(fixest)
{R_PREAMBLE}
model <- feols(lpacks ~ lrincome | lrprice ~ tdiff + rtax, data=df, vcov="iid")
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_2sls_iid.coef)
print("R SEs:", r_2sls_iid.se)

In [ ]:
# Note: polars_reg uses small-sample correction σ²=e'e/(n-k) while
# fixest uses asymptotic σ²=e'e/n, causing ~1.5% SE difference
print("=== 2SLS iid ===")
compare(res_2sls_iid, r_2sls_iid, rtol=1e-6, se_rtol=0.03, label="2SLS iid")

---
## Test 2: 2SLS with robust (HC1) standard errors

In [ ]:
# polars_reg
res_2sls_hc1 = pr.iv2sls(
    "lpacks ~ lrincome || lrprice ~ tdiff + rtax",
    data=df,
    vcov="HC1",
)
res_2sls_hc1.summary()

In [ ]:
# R: fixest feols with IV, heteroskedasticity-robust vcov
r_2sls_hc1 = run_r_regression(f'''
library(fixest)
{R_PREAMBLE}
model <- feols(lpacks ~ lrincome | lrprice ~ tdiff + rtax, data=df, vcov="hetero")
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_2sls_hc1.coef)
print("R SEs:", r_2sls_hc1.se)

In [ ]:
# Note: polars_reg uses small-sample correction σ²=e'e/(n-k) while
# fixest uses asymptotic σ²=e'e/n, causing ~1.5% SE difference
print("=== 2SLS HC1 ===")
compare(res_2sls_hc1, r_2sls_hc1, rtol=1e-6, se_rtol=0.03, label="2SLS HC1")

---
## Test 3: 2SLS clustered by state

In [ ]:
# polars_reg
res_2sls_cl = pr.iv2sls(
    "lpacks ~ lrincome || lrprice ~ tdiff + rtax",
    data=df,
    cluster=["state"],
)
res_2sls_cl.summary()

In [ ]:
# R: fixest feols with IV, clustered by state
r_2sls_cl = run_r_regression(f'''
library(fixest)
{R_PREAMBLE}
model <- feols(lpacks ~ lrincome | lrprice ~ tdiff + rtax, data=df, vcov=~state)
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_2sls_cl.coef)
print("R SEs:", r_2sls_cl.se)

In [ ]:
# Note: polars_reg uses small-sample correction σ²=e'e/(n-k) while
# fixest uses asymptotic σ²=e'e/n, causing ~1.5% SE difference
print("=== 2SLS Clustered (state) ===")
compare(res_2sls_cl, r_2sls_cl, rtol=1e-6, se_rtol=0.03, label="2SLS clustered")

---
## Test 4: LIML estimation

LIML is estimated using `AER::ivreg()` in R, since fixest does not support LIML.
Tolerance is relaxed to `rtol=2e-3` due to algorithmic differences.

In [ ]:
# polars_reg
res_liml = pr.liml(
    "lpacks ~ lrincome || lrprice ~ tdiff + rtax",
    data=df,
)
res_liml.summary()

In [ ]:
# R: AER ivreg with LIML
r_liml = run_r_regression(f'''
library(AER)
{R_PREAMBLE}
model <- ivreg(lpacks ~ lrincome + lrprice | lrincome + tdiff + rtax, data=df, method="LIML")
vcov_mat <- vcov(model)
{R_EXTRACT}
''')
print("R coefficients:", r_liml.coef)
print("R SEs:", r_liml.se)

In [ ]:
print("=== LIML ===")
compare(res_liml, r_liml, rtol=2e-3, label="LIML")

---
## Test 5: First-stage F-statistic

In [ ]:
# polars_reg first-stage F
print(f"polars_reg first-stage F: {res_2sls_iid.first_stage_f:.4f}")

In [ ]:
# R: extract first-stage F from fixest (stage=1 summary)
r_fsf = run_r_regression(f'''
library(fixest)
{R_PREAMBLE}
model <- feols(lpacks ~ lrincome | lrprice ~ tdiff + rtax, data=df, vcov="iid")
fs <- fitstat(model, "ivf")
f_val <- fs$ivf$stat

# Output as pseudo-regression result for parsing
cat("===RESULTS===\n")
cat("param,coef,se\n")
cat(sprintf("first_stage_F,%.15e,0\n", f_val))
cat("===META===\n")
cat(sprintf("N,%d\n", nobs(model)))
cat(sprintf("first_stage_F,%.15e\n", f_val))
''')

r_f_value = r_fsf.extra.get("first_stage_F", r_fsf.coef.get("first_stage_F", None))
print(f"R first-stage F:          {r_f_value:.4f}")

In [ ]:
print("=== First-stage F ===")
compare_scalar(res_2sls_iid.first_stage_f, r_f_value, "First-stage F", rtol=1e-4)

---
## Test 6: Weak instrument diagnostics

In [ ]:
# polars_reg weak instrument test
wit = pr.weak_instrument_test(res_2sls_iid, n_instruments=2)
print("Weak instrument test results:")
for k, v in wit.items():
    print(f"  {k}: {v}")

---
## Summary

| Test | polars_reg | R package | Tolerance | Status |
|------|-----------|-----------|-----------|--------|
| 2SLS iid | `iv2sls()` | `fixest::feols()` | 1e-6 | -- |
| 2SLS HC1 | `iv2sls(vcov="HC1")` | `fixest::feols(vcov="hetero")` | 1e-6 | -- |
| 2SLS clustered | `iv2sls(cluster=["state"])` | `fixest::feols(vcov=~state)` | 1e-6 | -- |
| LIML | `liml()` | `AER::ivreg(method="LIML")` | 2e-3 | -- |
| First-stage F | `.first_stage_f` | `fitstat(, "ivf")` | 1e-4 | -- |
| Weak instruments | `weak_instrument_test()` | -- | display only | -- |

Run all cells to fill in the Status column with PASS/FAIL.

In [ ]:
# Cleanup temp CSV
if os.path.exists(csv_path):
    os.remove(csv_path)
    print("Cleaned up temp CSV.")